In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
from scipy import stats
import seaborn as sns
import matplotlib.pyplot

### DATA OVERVIEW

In [ ]:
insurance_claims = pd.read_csv("../data/insurance_claims.csv")
print(insurance_claims.shape, '\n')
insurance_claims.head()

In [ ]:
insurance_claims.info()

In [ ]:
# the column '_c39' contained no non-null observations
insurance_claims = insurance_claims.drop(columns = ["_c39"])

In [ ]:
# no duplicates
insurance_claims.duplicated().sum()

In [ ]:
# 'umbrella_limit' contains negative values, could indicate a data quality problem
insurance_claims.describe().T

In [ ]:
# 'property_damage' and 'police_report_available' have '?' entries, may be missing values
# 1000 unique entries for 'incident_location', very spread so must be adjusted for modelling
insurance_claims.describe(include = "object").T

In [ ]:
# 91 missing values in 'authorities_contacted'
insurance_claims.isna().sum()

In [ ]:
insurance_claims.nunique()

### DATA CLEANING

#### UMBRELLA LIMIT 
One observation is negative. Checking the record, it is shown as 'N' for 'fraud_reported' suggesting a data entry error. This value will be converted to its absolute value. 

In [ ]:
insurance_claims["umbrella_limit"].value_counts()

In [ ]:
insurance_claims[insurance_claims["umbrella_limit"] < 0]

In [ ]:
insurance_claims["umbrella_limit"] = insurance_claims["umbrella_limit"].abs()

#### '?' ENTRIES
The columns 'collision_type', 'property_damage', and 'police_report_available' all contain '?' entries. As the frequency of these entries are high for each column, they are likely missing information. For this reason, they will not be removed but renamed to 'unknown' for clarity. 

In [ ]:
insurance_claims[insurance_claims == "?"].nunique()

In [ ]:
insurance_claims["collision_type"].value_counts()

In [ ]:
insurance_claims["property_damage"].value_counts()

In [ ]:
insurance_claims["police_report_available"].value_counts()

In [ ]:
insurance_claims["collision_type"] = insurance_claims["collision_type"].replace("?", "unknown")
insurance_claims["property_damage"] = insurance_claims["property_damage"].replace("?", "unknown")
insurance_claims["police_report_available"] = insurance_claims["police_report_available"].replace("?", "unknown")

#### MISSING VALUES
The column 'authorities_contacted' contains missing values. Checking the data, these are real missing values similar to the '?' entires. In this case, they will be replaced with 'unknown' for clarity.

In [ ]:
insurance_claims["authorities_contacted"].value_counts(dropna = False)

In [ ]:
insurance_claims["authorities_contacted"] = insurance_claims["authorities_contacted"].fillna("unknown")

#### FORMATTING
Looking at the dataset, there are no clear formatting inconsistencies. However, there are date columns which are not the datetime data type ~so this will be changed to datetime. 

In [ ]:
for col in insurance_claims.select_dtypes(include = "object"):
    print(col)
    print(insurance_claims[col].value_counts(), "\n")

In [ ]:
insurance_claims["incident_date"] = pd.to_datetime(insurance_claims["incident_date"])
insurance_claims["policy_bind_date"] = pd.to_datetime(insurance_claims["policy_bind_date"])

#### SAVING

In [ ]:
insurance_claims.info()

In [ ]:
insurance_claims.to_parquet("../data/clean_insurance_claims.parquet")